Import thư viện và đặt số nút con, threshold

In [1]:
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

branching_factor, threshold = 10, 0.5
columns = ["Grocery", "Milk"]

Thêm và đọc dữ liệu

In [2]:
data = pd.read_csv("data/wholesale_customers.csv")
example_data = pd.read_csv("data/minh_hoa_birch.csv")
print("Wholesale customers:", data.shape)
print(data.head())
print("\nMinh họa Birch:", example_data.shape)
print(example_data.head())
print("\nThống kê hai đặc trưng:")
print(data[columns].describe().round(2))

Wholesale customers: (440, 8)
   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185

Minh họa Birch: (20, 2)
   Grocery  Milk
0        1     2
1        1     4
2        2     3
3        2     2
4        5     8

Thống kê hai đặc trưng:
        Grocery      Milk
count    440.00    440.00
mean    7951.28   5796.27
std     9503.16   7380.38
min        3.00     55.00
25%     2153.00   1533.00
50%     4755.50   3627.00
75%    10655.75   7190.25
max    92780.00  73498.00


Tiền xử lý: Chuẩn hóa Z-Score

In [3]:
raw_data = data[columns].to_numpy(dtype=float)
X = StandardScaler().fit_transform(raw_data)
print("Các cột:", columns)
print("Trung bình sau chuẩn hóa:", X.mean(axis=0).round(4))
print("Độ lệch chuẩn sau chuẩn hóa:", X.std(axis=0).round(4))

Các cột: ['Grocery', 'Milk']
Trung bình sau chuẩn hóa: [-0.  0.]
Độ lệch chuẩn sau chuẩn hóa: [1. 1.]


Cài đặt CF, CF Entry, CF Node

In [ ]:
class ClusteringFeature:
    def __init__(self, n, ls, ss):
        self.n = n
        self.ls = ls
        self.ss = ss

    def __add__(self, other):
        return ClusteringFeature(self.n + other.n, self.ls + other.ls,
                                 self.ss + other.ss)

    def centroid(self):
        return self.ls / self.n

    def radius(self):
        center = self.centroid()
        radius_squared = self.ss / self.n - np.dot(center, center)
        return float(np.sqrt(max(radius_squared, 0.0)))

    def to_dict(self):
        return {"N": self.n, "LS": self.ls.tolist(), "SS": self.ss,
                "centroid": self.centroid().tolist(), "radius": self.radius()}


class CFEntry:
    def __init__(self, cf, child=None):
        self.cf = cf
        self.child = child


class CFNode:
    def __init__(self, is_leaf, entries=None):
        self.is_leaf = is_leaf
        self.entries = [] if entries is None else entries

    def summary(self):
        total = self.entries[0].cf
        for entry in self.entries[1:]:
            total = total + entry.cf
        return total

Cài đặt CF Tree